In [2]:
%load_ext autoreload
%autoreload 2

# LSTM-Based Human Activity Recognition / LSTMを用いたHuman Activity Recognition

This notebook investigates the use of a **Long Short-Term Memory (LSTM)** network for Human Activity Recognition (HAR) using 2D human pose keypoints. Unlike the traditional machine learning models explored in the previous notebooks, the LSTM is capable of learning temporal dependencies directly from sequential pose data.

The objective is to evaluate whether a sequence-based deep learning model can improve recognition performance compared to feature-engineered machine learning approaches under the same Leave-One-Subject-Out (LOSO) evaluation protocol.

---

# LSTMを用いたHuman Activity Recognition

本ノートブックでは、2次元姿勢キーポイントを用いた Human Activity Recognition（HAR）に対して、**Long Short-Term Memory（LSTM）** ネットワークを適用する。これまでのノートブックで評価した従来の機械学習モデルとは異なり、LSTMは時系列データから時間的な依存関係を直接学習することができる。

本実験の目的は、同じ Leave-One-Subject-Out（LOSO）評価手法のもとで、系列データを扱う深層学習モデルが特徴量ベースの機械学習手法より高い認識性能を示すかを評価することである。

# Imports

In [3]:
# ==========================================================
# Import project modules / プロジェクトモジュールの読み込み
# ==========================================================

# Load utility functions for reading datasets and formatting notebook output
# データセットの読み込みとノートブック出力を整形するユーティリティ
from har_utils.data import (
    get_subject,
    print_clean_header
)

# Load global configuration values shared across the project
# プロジェクト全体で使用する設定値を読み込む
from har_utils.config import FILE_NAME_SUFFIX

# Load preprocessing functions for data cleaning and feature preparation
# データクリーニングと特徴量生成前処理を行う関数
from har_utils.preprocessing import (
    build_loso_sequence_datasets,
    preprocessing_pipeline,
    preprocess_all_subjects,
    scale_sequences
)

# Load machine learning model constructors
# 機械学習モデルを生成する関数
from har_utils.model import model_random_forest


# Load evaluation utilities for training and performance analysis
# モデルの学習および性能評価を行う関数
from har_utils.analysis import (
    preprocess_evaluate_loso_supervised,
    run_optuna_study,
    evaluate_loso,
    get_feature_importance,
    filter_features_by_importance
)

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from sklearn.preprocessing import LabelEncoder

from har_utils.analysis import evaluate_lstm_loso

/Users/sozolab/Documents/GitHub/pose-based-human-activity-recognition/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Load Data

In [4]:
# Load all subject datasets into a dictionary for easy access
# 各被験者のデータセットを辞書に読み込み、一括管理する
all_subject = {}

for suffix in FILE_NAME_SUFFIX:

    # Read the current subject dataset from the CSV file
    # 現在の被験者データをCSVファイルから読み込む
    all_subject[suffix] = get_subject(suffix)

    # Display the dataset dimensions to verify successful loading
    # データが正しく読み込まれたか、データサイズを確認する
    print(f"Subject_{suffix} Shape: {all_subject[suffix].shape}")

Subject_1 Shape: (76456, 35)
Subject_2 Shape: (74638, 35)
Subject_3 Shape: (118087, 35)
Subject_5 Shape: (75981, 35)


# Generate Frame Sequence for LSTM

In [5]:
# Run pipeline to FRAME LEVEL, not windowed
# Note return_stage='features' and drop_bad_shoulder_frames=True
all_subject_frames = {}
for suffix in FILE_NAME_SUFFIX:
    all_subject_frames[suffix] = preprocessing_pipeline(
        all_subject[suffix],
        return_stage='features',
        drop_bad_shoulder_frames=False
    )

In [6]:
# Build LSTM sequences
loso_sequences = build_loso_sequence_datasets(
    all_subject_frames,
    sequence_length=90,
    stride=10
)


Subject 1:
  Sequences created : 7177
  Sequence shape    : (90, 118)  (timesteps × features)
  Classes encoded   : [np.str_('Attacking'), np.str_('Biting'), np.str_('Eating snacks'), np.str_('Head banging'), np.str_('Sitting quietly'), np.str_('Throwing things'), np.str_('Using phone'), np.str_('Walking')]

Subject 2:
  Sequences created : 7005
  Sequence shape    : (90, 118)  (timesteps × features)
  Classes encoded   : [np.str_('Attacking'), np.str_('Biting'), np.str_('Eating snacks'), np.str_('Head banging'), np.str_('Sitting quietly'), np.str_('Throwing things'), np.str_('Using phone'), np.str_('Walking')]

Subject 3:
  Sequences created : 10455
  Sequence shape    : (90, 118)  (timesteps × features)
  Classes encoded   : [np.str_('Attacking'), np.str_('Biting'), np.str_('Eating snacks'), np.str_('Head banging'), np.str_('Sitting quietly'), np.str_('Throwing things'), np.str_('Using phone'), np.str_('Walking')]

Subject 5:
  Sequences created : 7054
  Sequence shape    : (90, 118

In [7]:
# Verify shapes
for sid, data in loso_sequences.items():
    print(f"Subject {sid}: X={data['X'].shape}, y={data['y'].shape}")

Subject 1: X=(7177, 90, 118), y=(7177,)
Subject 2: X=(7005, 90, 118), y=(7005,)
Subject 3: X=(10455, 90, 118), y=(10455,)
Subject 5: X=(7054, 90, 118), y=(7054,)


# Label encoding consistency across subjects.

In [8]:
# Fit one encoder on all possible labels across all subjects
all_labels = []
for data in loso_sequences.values():
    all_labels.extend(data['encoder'].classes_.tolist())

shared_encoder = LabelEncoder()
shared_encoder.fit(list(set(all_labels)))

print("Shared class mapping:")
for i, cls in enumerate(shared_encoder.classes_):
    print(f"   {i} -> {cls}")

Shared class mapping:
   0 -> Attacking
   1 -> Biting
   2 -> Eating snacks
   3 -> Head banging
   4 -> Sitting quietly
   5 -> Throwing things
   6 -> Using phone
   7 -> Walking


In [9]:
for subject_id, data in loso_sequences.items():
    # Get original string labels back from the per-subject encoder
    string_labels = data['encoder'].inverse_transform(data['y'])
    # Re-encode with the shared encoder
    data['y'] = shared_encoder.transform(string_labels)

print("All subjects re-encoded with shared encoder.")

All subjects re-encoded with shared encoder.


# Feature Scaling & LSTM LOSO Evaluation

The scaler must be fitted on training data only and applied to test data. Never fit on test data — that would leak information.

In [11]:
lstm_results = evaluate_lstm_loso(
    loso_sequences=loso_sequences,
    shared_encoder=shared_encoder,
    n_epochs=20,
    batch_size=32,
    lstm_units=64,
    sequence_length=90
)


────────────────────────────────────────────────────
  Fold — Test Subject: 1  |  Train: ['2', '3', '5']
────────────────────────────────────────────────────


/Users/sozolab/Documents/GitHub/pose-based-human-activity-recognition/.venv/lib/python3.12/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


  Trained for 4 epochs

  Class                  Precision    Recall        F1
  ────────────────────── ───────── ───────── ─────────
  Attacking                   0.32      0.23      0.27
  Biting                      0.73      0.71      0.72
  Eating snacks               0.67      0.49      0.56
  Head banging                0.93      0.53      0.68
  Sitting quietly             0.45      0.50      0.47
  Throwing things             0.09      0.34      0.15
  Using phone                 0.00      0.00      0.00
  Walking                     0.75      0.99      0.86

  Macro F1: 0.464

────────────────────────────────────────────────────
  Fold — Test Subject: 2  |  Train: ['1', '3', '5']
────────────────────────────────────────────────────


/Users/sozolab/Documents/GitHub/pose-based-human-activity-recognition/.venv/lib/python3.12/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


  Trained for 7 epochs

  Class                  Precision    Recall        F1
  ────────────────────── ───────── ───────── ─────────
  Attacking                   0.75      0.20      0.31
  Biting                      0.61      0.99      0.75
  Eating snacks               0.80      0.70      0.75
  Head banging                0.90      0.97      0.93
  Sitting quietly             0.58      0.95      0.72
  Throwing things             0.28      0.38      0.32
  Using phone                 0.83      0.37      0.52
  Walking                     0.98      0.93      0.96

  Macro F1: 0.657

────────────────────────────────────────────────────
  Fold — Test Subject: 3  |  Train: ['1', '2', '5']
────────────────────────────────────────────────────


/Users/sozolab/Documents/GitHub/pose-based-human-activity-recognition/.venv/lib/python3.12/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


  Trained for 4 epochs

  Class                  Precision    Recall        F1
  ────────────────────── ───────── ───────── ─────────
  Attacking                   0.64      0.74      0.68
  Biting                      0.95      0.14      0.24
  Eating snacks               0.24      0.08      0.11
  Head banging                0.04      0.00      0.00
  Sitting quietly             0.97      0.37      0.54
  Throwing things             0.09      0.74      0.15
  Using phone                 0.30      0.36      0.33
  Walking                     0.78      0.95      0.85

  Macro F1: 0.364

────────────────────────────────────────────────────
  Fold — Test Subject: 5  |  Train: ['1', '2', '3']
────────────────────────────────────────────────────


/Users/sozolab/Documents/GitHub/pose-based-human-activity-recognition/.venv/lib/python3.12/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


  Trained for 4 epochs

  Class                  Precision    Recall        F1
  ────────────────────── ───────── ───────── ─────────
  Attacking                   0.85      0.30      0.45
  Biting                      0.00      0.00      0.00
  Eating snacks               0.30      0.81      0.44
  Head banging                1.00      0.77      0.87
  Sitting quietly             0.53      0.24      0.34
  Throwing things             0.57      0.88      0.70
  Using phone                 0.00      0.00      0.00
  Walking                     0.93      1.00      0.96

  Macro F1: 0.469

════════════════════════════════════════════════════
  LSTM LOSO RESULTS SUMMARY
════════════════════════════════════════════════════
  Subject        Macro F1
  ──────────── ──────────
  1                 0.464
  2                 0.657
  3                 0.364
  5                 0.469
  ──────────── ──────────
  Mean              0.488
  Std               0.106
══════════════════════════════════════